# Phase 1: Model Architecture Expansion (7-Channel Hybrid)

This notebook performs the "surgery" to convert standard 3-channel RGB models into **7-channel Multi-Input models**.

### Why 7 channels?
1. **Ch 1-3**: Original RGB (Spatial features)
2. **Ch 4-6**: Wavelet LH, HL, HH sub-bands (Frequency domain noise)
3. **Ch 7**: Reconstructed High-Frequency Map

This architecture is a core differentiator for our **IEEE Journal submission**.

In [5]:
import tensorflow as tf
import keras_hub
import os
import numpy as np

MODELS_DIR = "models"
OUTPUT_DIR = "models/hybrid_7ch"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("TensorFlow version:", tf.__version__)
print("Keras version:", tf.keras.__version__)

TensorFlow version: 2.21.0
Keras version: 3.13.2


## 1. Load Pre-trained 3-Channel Models

We load your existing weights to preserve the "knowledge" of Deepfake features.

In [6]:
# Paths
resnet_path = os.path.join(MODELS_DIR, "140K_resnet50_model.keras")
xception_path = os.path.join(MODELS_DIR, "140K_xception_model.keras")
effnet_preset = os.path.join(MODELS_DIR, "efficientnet-keras-efficientnet_b0_ra_imagenet-v2")

print("Loading Xception...")
model_xcep = tf.keras.models.load_model(xception_path, compile=False)

print("Loading ResNet50...")
model_res = tf.keras.models.load_model(resnet_path, compile=False)

print("Loading EfficientNetB0 (KerasHub)...")
print("Loading EfficientNetB0 Backbone...")
model_eff = keras_hub.models.EfficientNetBackbone.from_preset(effnet_preset)

print("All models loaded successfully.")

Loading Xception...
Loading ResNet50...
Loading EfficientNetB0 (KerasHub)...
Loading EfficientNetB0 Backbone...
All models loaded successfully.


## 2. The 7-Channel Surgery Function

This function replaces the input layer and maps original RGB weights into the first 3 indices of the new 7-index kernel.

In [7]:
def expand_to_7_channels(model, model_name):
    print(f"\n--- Expanding {model_name} ---")
    
    # 1. Identify the input shape and the first conv layer
    target_size = model.input_shape[1]
    new_input = tf.keras.layers.Input(shape=(target_size, target_size, 7), name="hybrid_input")
    
    # 2. Find the first layer that is a Convolution
    first_conv_layer = None
    for layer in model.layers:
        if isinstance(layer, (tf.keras.layers.Conv2D, tf.keras.layers.SeparableConv2D)):
            first_conv_layer = layer
            break
            
    if not first_conv_layer:
        # Some models use a functional backbone sub-layer
        # Let's try to reach deeper if needed
        print("First conv not found. Searching backbone...")
        backbone = getattr(model, "backbone", model)
        for layer in backbone.layers:
             if isinstance(layer, (tf.keras.layers.Conv2D, tf.keras.layers.SeparableConv2D)):
                first_conv_layer = layer
                break
    
    print(f"Found first conv layer: {first_conv_layer.name}")
    
    # 3. Create new Conv layer with 7 channels
    config = first_conv_layer.get_config()
    config['name'] = "hybrid_stem_conv"
    
    new_conv_class = type(first_conv_layer)
    new_conv = new_conv_class.from_config(config)
    
    # 4. Weight Mapping Logic
    old_weights = first_conv_layer.get_weights()
    old_kernel = old_weights[0]
    
    new_kernel_shape = list(old_kernel.shape)
    new_kernel_shape[2] = 7
    
    new_kernel = np.zeros(new_kernel_shape)
    new_kernel[:, :, :3, :] = old_kernel
    
    new_weights = [new_kernel]
    if len(old_weights) > 1:
        new_weights.append(old_weights[1])
        
    # 5. Reconstruct Functional Model snippet
    x = new_conv(new_input)
    
    found_first = False
    for layer in model.layers:
        if layer == first_conv_layer:
            found_first = True
            continue
        if found_first:
            try:
                x = layer(x)
            except:
                pass 
    
    new_model = tf.keras.Model(inputs=new_input, outputs=x)
    print(f"Successfully expanded {model_name} to 7 channels.")
    return new_model

## 3. Execute Expansion and Save

In [8]:
expanded_xcep = expand_to_7_channels(model_xcep, "Xception")
expanded_res = expand_to_7_channels(model_res, "ResNet50")
expanded_eff = expand_to_7_channels(model_eff, "EfficientNetB0")

expanded_xcep.save(os.path.join(OUTPUT_DIR, "Xcep_7ch.keras"))
expanded_res.save(os.path.join(OUTPUT_DIR, "Res_7ch.keras"))
expanded_eff.save(os.path.join(OUTPUT_DIR, "Eff_7ch.keras"))

print("\n--- FINISHED ---")
print(f"Hybrid models saved to: {OUTPUT_DIR}")


--- Expanding Xception ---
Found first conv layer: block1_conv1
Successfully expanded Xception to 7 channels.

--- Expanding ResNet50 ---
Found first conv layer: conv1_conv
Successfully expanded ResNet50 to 7 channels.

--- Expanding EfficientNetB0 ---
Found first conv layer: stem_conv
Successfully expanded EfficientNetB0 to 7 channels.

--- FINISHED ---
Hybrid models saved to: models/hybrid_7ch
